# V10 - Consultas com GraphQL e SDK

**Objetivo:** criar um data model de teste com uma instancia, introspeccionar o schema, executar uma consulta GraphQL real e mostrar a intencao equivalente pelo SDK de instancias.

**GraphQL vs SDK:**
- **GraphQL** e otimo para o consumidor escolher exatamente os campos e navegar relacoes.
- **SDK de instancias** e melhor para pipelines, paginacao programatica e operacoes em lote.

In [ ]:
# Carrega a configuracao local e cria o cliente CDF. / Load local configuration and create the CDF client.
import os
from getpass import getpass
from pathlib import Path

from dotenv import load_dotenv
from cognite.client import ClientConfig, CogniteClient
from cognite.client.credentials import OAuthClientCredentials, Token

env_file = Path('.env')
pkg_root = next(
    (p for p in (Path.cwd(), *Path.cwd().parents) if (p / '.env.example').exists()),
    Path.cwd(),
)

if env_file.exists():
    load_dotenv(env_file)
else:
    for name in ('.env', '.env.example'):
        candidate = pkg_root / name
        if candidate.exists():
            load_dotenv(candidate)
            break

if not os.environ.get('CDF_CLUSTER'):
    os.environ['CDF_CLUSTER'] = input('CDF_CLUSTER: ').strip()
if not os.environ.get('CDF_PROJECT'):
    os.environ['CDF_PROJECT'] = input('CDF_PROJECT: ').strip()

base_url = os.environ.get('CDF_URL', '').rstrip('/') or f"https://{os.environ['CDF_CLUSTER']}.cognitedata.com"
oauth_ready = env_file.exists() and all(
    os.environ.get(key) for key in ('IDP_TOKEN_URL', 'IDP_CLIENT_ID', 'IDP_CLIENT_SECRET')
)
if oauth_ready:
    scopes = os.environ.get('IDP_SCOPES', '').replace(',', ' ').split() or [f'{base_url}/.default']
    audience = os.environ.get('IDP_AUDIENCE', '')
    credentials = OAuthClientCredentials(
        token_url=os.environ['IDP_TOKEN_URL'],
        client_id=os.environ['IDP_CLIENT_ID'],
        client_secret=os.environ['IDP_CLIENT_SECRET'],
        scopes=scopes,
        **({'audience': audience} if audience else {}),
    )
else:
    # Sem .env nesta pasta: usa Bearer Token.
    bearer_token = (os.environ.get('CDF_BEARER_TOKEN') or getpass('CDF_BEARER_TOKEN: ')).strip()
    if bearer_token.lower().startswith('bearer '):
        bearer_token = bearer_token[7:].strip()
    credentials = Token(bearer_token)

client = CogniteClient(ClientConfig(
    client_name='radix-cdf-onboarding-v10',
    base_url=base_url,
    project=os.environ['CDF_PROJECT'],
    credentials=credentials,
))

## 1. Criar data model de teste + 1 instancia

In [ ]:
from uuid import uuid4
from cognite.client.data_classes.data_modeling import (
    ContainerApply, ContainerId, DataModelApply, MappedPropertyApply, NodeApply,
    NodeId, NodeOrEdgeData, SpaceApply, Text, ViewApply, ViewId,
)

run = uuid4().hex[:8]
sp = f'sp_ur_training_v10_{run}'
container_id = ContainerId(sp, 'Equipment')
view_id = ViewId(sp, 'Equipment', 'v1')
model_id = (sp, 'EquipmentModel', 'v1')

client.data_modeling.spaces.apply(SpaceApply(space=sp, name='UR training - V10'))
client.data_modeling.containers.apply(ContainerApply._load({
    'space': sp, 'externalId': container_id.external_id,
    'properties': {'name': {'type': Text().dump(), 'nullable': False}},
    'usedFor': 'node',
}))
client.data_modeling.views.apply(ViewApply(
    space=sp, external_id=view_id.external_id, version=view_id.version,
    properties={'name': MappedPropertyApply(container=container_id, container_property_identifier='name')},
))
client.data_modeling.data_models.apply(DataModelApply(
    space=sp, external_id='EquipmentModel', version='v1', views=[view_id],
))
client.data_modeling.instances.apply(nodes=NodeApply(
    space=sp, external_id='eq-001',
    sources=[NodeOrEdgeData(source=view_id, properties={'name': 'Compressor 01'})],
))
print('data model e instancia criados em', sp)

## 2. Introspeccao do schema (GraphQL)
Descobre os tipos disponiveis no modelo.

In [ ]:
from cognite.client.exceptions import CogniteAPIError

intro = client.data_modeling.graphql.query(id=model_id, query='{ __schema { queryType { name } types { name kind } } }')
type_names = [t['name'] for t in intro['__schema']['types'] if not t['name'].startswith('__')]
print('tipos no modelo (amostra):', type_names[:15])

## 3. Query GraphQL real
Seleciona campos do tipo `Equipment`. Protegida por try/except (nomes de campo dependem do modelo).

In [ ]:
gql = '''
{
  listEquipment(first: 5) {
    items { externalId name }
  }
}
'''
try:
    data = client.data_modeling.graphql.query(id=model_id, query=gql)
    print('resultado GraphQL:', data)
except CogniteAPIError as exc:
    print('Consulta de campos falhou (ajuste os nomes conforme a introspeccao acima):', exc.code)

## 4. Mesma intencao pelo SDK de instancias
O equivalente programatico, ideal para pipelines.

In [ ]:
import pandas as pd

nodes = client.data_modeling.instances.list(instance_type='node', sources=view_id, space=sp, limit=5)
sdk_df = nodes.to_pandas()
sdk_df

## Mini-exercicio
- Use a introspeccao para descobrir outro campo e adicione-o a query GraphQL.
- Compare o numero de chamadas/linhas de codigo entre GraphQL e SDK para o mesmo resultado.

## 5. Limpeza idempotente

In [ ]:
assert sp.startswith('sp_ur_training_v10_')
client.data_modeling.instances.delete(nodes=NodeId(sp, 'eq-001'))
client.data_modeling.data_models.delete(model_id)
client.data_modeling.views.delete(view_id)
client.data_modeling.containers.delete(container_id)
client.data_modeling.spaces.delete(sp)
print('space_still_exists:', client.data_modeling.spaces.retrieve(sp) is not None)